In [ ]:
import warnings
from pathlib import Path
from shutil import rmtree
from json import dumps
from typing import Literal
from pickle import Pickler
from gzip import open as gz_open
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, minmax_scale
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector


RANDOM_STATE = 12345

# Data

In [ ]:
def get_sampling_strategy(y):
    class_names, counts = np.unique(y, return_counts=True)
    minority = int(counts.min())
    if len(class_names) < 3:
        target_size = min(minority, 5000)
        return {cls_name: target_size for cls_name in class_names}
    if len(class_names) == 8:
        return {cls_name: minority for cls_name in class_names}
    return {
        cls_name: min(int(counts[i]), 50 * minority)
        for i, cls_name in enumerate(class_names)
    }


def get_class_weight(y):
    sample_strategy = get_sampling_strategy(y)
    num_classes = len(sample_strategy)
    total_samples = sum(sample_strategy.values())
    return {
        cls_name: round(total_samples / (num_classes * count), 2)
        for cls_name, count in sample_strategy.items()
    }


def get_param_grid(num_classes):
    param_grid = {
        "model__criterion": ["gini", "entropy"],
        "model__max_features": [0.25, 0.5],
        "model__min_samples_leaf": [3, 5, 7],
    }
    if num_classes < 3:
        param_grid["model__n_estimators"] = [100, 150, 200]
        param_grid["model__max_leaf_nodes"] = [300, 600, 900]
    elif num_classes == 8:
        param_grid["model__n_estimators"] = [75, 100, 125]
        param_grid["model__max_leaf_nodes"] = [600, 900, 1200]
    else:
        param_grid["model__n_estimators"] = [30, 40, 50]
        param_grid["model__max_leaf_nodes"] = [1200, 1800, 2400]
        param_grid["model__max_features"] = [0.4, 0.6]
    return param_grid


def prepare_experiment(
    which: Literal["full", "iot", "control", "infra"],
    target: Literal["binary", "grouped", "all"],
    overwrite: bool = False,
):
    experiment = {}
    dataset_path = Path("../datasense/dataset/")
    for split in ["train", "test"]:
        split_data = dataset_path / f"datasense_{which}_{split}_1sec.parquet"
        split_data = pd.read_parquet(split_data)
        if target == "binary":
            y_split = split_data["label1"]
        elif target == "grouped":
            y_split = split_data["label2"]
        else:
            y_split = split_data["label4"]
        experiment[split] = (split_data, y_split)
    experiment["num_classes"] = experiment["train"][1].nunique()
    number_selector = make_column_selector(dtype_include="number")
    bool_selector = make_column_selector(dtype_include="bool")
    if which == "full":
        experiment["transformers"] = make_column_transformer(
            ("passthrough", number_selector),
            ("passthrough", bool_selector),
            (OneHotEncoder(sparse_output=False), ["device_type"]),
            remainder="drop",
            verbose_feature_names_out=False,
        )
    else:
        experiment["transformers"] = make_column_transformer(
            ("passthrough", number_selector),
            ("passthrough", bool_selector),
            remainder="drop",
            verbose_feature_names_out=False,
        )
    results_dir = Path("results")
    results_dir /= f"random_forest_{which}_{target}"
    if results_dir.is_dir():
        if overwrite:
            rmtree(results_dir, ignore_errors=True)
        else:
            raise RuntimeError("Experiment already executed and overwrite set to False")
    results_dir.mkdir(exist_ok=True, parents=True)
    cache_dir = results_dir / ".cache"
    cache_dir.mkdir()
    experiment["results_dir"] = results_dir
    experiment["cache_dir"] = str(cache_dir)
    return experiment

# Experiment

In [ ]:
experiment = prepare_experiment(which="full", target="all", overwrite=True)

X_train, y_train = experiment["train"]
X_test, y_test = experiment["test"]
results_dir = experiment["results_dir"]
cache_dir = experiment["cache_dir"]

## Training pipe

In [ ]:
train_pipe = Pipeline(
    steps=[
        ("transformer", experiment["transformers"]),
        (
            "model",
            BalancedRandomForestClassifier(
                bootstrap=False,
                replacement=False,
                sampling_strategy=get_sampling_strategy,
                class_weight=get_class_weight(y_train),
                random_state=RANDOM_STATE,
            ),
        ),
    ],
    memory=cache_dir,
    verbose=False,
)

## Grid seach cross-validation

In [ ]:
param_grid = get_param_grid(experiment["num_classes"])

In [ ]:
grid = GridSearchCV(
    estimator=train_pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    n_jobs=4,
    refit=True,
    cv=4,
    verbose=3,
    return_train_score=True,
)

with warnings.catch_warnings():
    warnings.filterwarnings(action="ignore", message=r"^A worker stopped while")
    grid.fit(X_train, y_train)

trained_pipe = grid.best_estimator_
trained_model = trained_pipe["model"]

## Cross-validation results

In [ ]:
cv_results = pd.DataFrame(grid.cv_results_)
cv_results.sort_values(by="rank_test_score", inplace=True)
cv_results.to_csv(results_dir / "train_cv_results.csv", index=False)

cv_results.head(10)

## Feature importance

In [ ]:
feature_names = trained_pipe["transformer"].get_feature_names_out()

feature_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": trained_pipe["model"].feature_importances_,
})
feature_importances.sort_values(
    by="importance", ascending=False,
    ignore_index=True, inplace=True,
)
feature_importances["importance_norm"] = minmax_scale(
    feature_importances["importance"]
)

feature_importances.to_csv(
    results_dir / "feature_importances.csv", index=False
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 16))

feature_importances["importance"].plot(kind="barh", ax=ax)

ax.invert_yaxis()
ax.set_yticks(
    ticks=range(feature_importances.shape[0]),
    labels=feature_importances["feature"],
    fontsize=6,
)
ax.set_title("Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.savefig(results_dir / "feature_importances.png", dpi=300)
plt.show()

# Model evaluations

## Train-split evaluations

In [ ]:
y_proba = trained_pipe.predict_proba(X_train)

train_results = pd.DataFrame(
    data=y_proba,
    columns=trained_pipe.classes_,
    dtype="float32",
)

train_results.to_parquet(
    results_dir / "train_predict_proba.parquet",
    index=False,
    compression="gzip",
)

## Test-split evaluations

In [ ]:
y_proba = trained_pipe.predict_proba(X_test)

test_results = pd.DataFrame(
    data=y_proba,
    columns=trained_pipe.classes_,
    dtype="float32",
)

test_results.to_parquet(
    results_dir / "test_predict_proba.parquet",
    index=False,
    compression="gzip",
)

# Model summary and persistency

## Model persistency

In [ ]:
trained_model.samplers_ = []
trained_model.pipelines_ = []

pipe_persist_path = results_dir / "pipe.pickle.gz"

with gz_open(pipe_persist_path, mode="wb") as pf:
    Pickler(pf, protocol=5).dump(trained_pipe)

## Model summary

In [ ]:
total_n_nodes = sum(dtree.tree_.node_count for dtree in trained_model.estimators_)
total_n_leaves = sum(dtree.tree_.n_leaves for dtree in trained_model.estimators_)
total_depths = sum(dtree.tree_.max_depth for dtree in trained_model.estimators_)

model_summary = {
    "best_parameters": grid.best_params_,
    "all_parameters": trained_model.get_params(),
    "forest_structure": {
        "total_n_nodes": total_n_nodes,
        "mean_n_nodes": round(total_n_nodes / len(trained_model.estimators_), 2),
        "mean_n_leafs": round(total_n_leaves / len(trained_model.estimators_), 2),
        "mean_depth": round(total_depths / len(trained_model.estimators_), 2),
    }
}

model_summary = dumps(model_summary, indent=2, default=str)
(results_dir / "model_summary.json").write_text(model_summary)
print("MODEL SUMMARY:", model_summary, sep="\n")